# H2 · UT-HAR veri analizi ve oznitelikler
**Amac:** Veri setini incelemek, hangi ozniteliklerin (feature) oldugunu ve ne ise yaradiklarini gostermek.
Sonucta klasik makine ogrenmesi algoritmalari icin 910 oznitelik uretilip Drive'a kaydedilir
(`data/ozellik_uthar.npz`); ekipteki herkes `H2_ML_sablon.ipynb` ile bu dosyayi kullanir.

Sirayla calistir: **Runtime > Run all**. Grafiklerin altindaki notlar rapora yazilacak yorumlardir.

In [ ]:
# KURULUM: Colab'da Drive'i baglar, depoyu GitHub'dan ceker. (Yerelde test icin YSA_PROJE_YOLU kullanilir.)
import os, sys, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run("rm -rf /content/ysa-wifi-csi && git clone -q https://github.com/fat1hkaya/ysa-wifi-csi /content/ysa-wifi-csi",
                   shell=True, check=True)
    sys.path.insert(0, "/content/ysa-wifi-csi/src")
except ImportError:
    sys.path.insert(0, os.environ.get("YSA_SRC", "../src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import ortak, ozellik
from ortak import SINIFLAR, DUSME
print("Veri yolu:", ortak.VERI_YOLU)

## 1. Verinin yapisi
Her ornek bir **matris**: 250 zaman adimi (Wi-Fi paketi) x 90 kanal.
90 kanal = **3 alici anten x 30 alt tasiyici (subcarrier)**; her deger o anda o kanalin **CSI genligi**dir
(sinyalin odada ne kadar zayiflayip yansidigi). Tablo halinde isimli sutunlar yoktur; ham oznitelikler
bu 22.500 sayidir. Kanal adlarini biz verdik: `a1_t01` = anten 1, alt tasiyici 1.

In [ ]:
v = ortak.uthar_yukle()          # normalizasyon yalnizca egitim min/max'i ile
for k in ["X_train", "X_val", "X_test"]:
    x = v[k]
    print(f"{k:8s} sekil={x.shape}  tip={x.dtype}  NaN={np.isnan(x).sum()}  inf={np.isinf(x).sum()}  "
          f"min={x.min():.3f}  maks={x.max():.3f}  ort={x.mean():.3f}")
print("Toplam ornek:", sum(len(v[k]) for k in ["y_train", "y_val", "y_test"]))
print("Ornek basina ham deger:", 250 * 90)
kanallar = ozellik.kanal_adlari()
print("Ilk/son kanal:", kanallar[:3], "...", kanallar[-3:])

## 2. Etiketler ve sinif dagilimi
0 yatma · 1 **dusme** · 2 yurume · 3 egilip alma · 4 kosma · 5 oturma · 6 kalkma.
Dusme ornegi azdir (~%9) -> dengesiz veri. Bu yuzden **accuracy yetmez**; macro-F1 ve dusme recall'u kullanacagiz,
modellerde `class_weight="balanced"` acik olacak.

In [ ]:
import pandas as pd
tablo = pd.DataFrame({b: np.bincount(v[f"y_{b}"], minlength=7) for b in ["train", "val", "test"]},
                     index=[f"{i} {s}" for i, s in enumerate(SINIFLAR)])
tablo["toplam"] = tablo.sum(1); tablo["oran_%"] = (100 * tablo["toplam"] / tablo["toplam"].sum()).round(1)
display(tablo)
tablo[["train", "val", "test"]].plot.bar(figsize=(8, 3.5), rot=30, title="Sinif dagilimi")
plt.ylabel("ornek sayisi"); plt.tight_layout(); plt.show()

## 3. Sinif ortalamasi CSI desenleri
Her sinifin tum orneklerinin ortalamasi. Yatay eksen zaman, dikey eksen 90 kanal.
Ortalama alininca hareketin kendisi silinir; geriye sinifin tipik "parmak izi" kalir.

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(16, 6.5)); ax = ax.ravel()
for c in range(7):
    ort = v["X_train"][v["y_train"] == c].mean(0)
    ax[c].imshow(ort.T, aspect="auto", cmap="viridis"); ax[c].set_title(f"{c} {SINIFLAR[c]}")
    ax[c].set_xlabel("zaman"); ax[c].set_ylabel("kanal")
ax[7].axis("off"); plt.tight_layout(); plt.show()

## 4. Tek kanalda zaman serisi
Ayni kanalin (anten 1, alt tasiyici 15) her siniftan bir ornekte zaman icindeki degisimi.
**Beklenen:** yatma/oturma duz, kosma/yurume hizli salinimli, dusmede kisa ve sert bir sicrama.

In [ ]:
k = kanallar.index("a1_t15")
fig, ax = plt.subplots(7, 1, figsize=(12, 11), sharex=True)
for c in range(7):
    i = np.where(v["y_train"] == c)[0][0]
    ax[c].plot(v["X_train"][i, :, k], lw=1); ax[c].set_ylabel(SINIFLAR[c], rotation=0, ha="right")
ax[-1].set_xlabel("zaman adimi"); plt.tight_layout(); plt.show()

## 5. Anten ve alt tasiyici profili
**Sol:** her alt tasiyicinin ortalama genligi (3 anten ayri cizgi) -> antenler farkli konumda oldugu icin farkli seviyeler.
**Sag:** her kanalin zaman icindeki **standart sapmasi**, sinif bazinda -> hareketli siniflarda yuksek; hangi
kanallarin harekete daha duyarli oldugunu gosterir.

In [ ]:
X = v["X_train"]; y = v["y_train"]
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
prof = X.mean((0, 1)).reshape(3, 30)
for a_ in range(3): ax[0].plot(range(1, 31), prof[a_], marker=".", label=f"anten {a_ + 1}")
ax[0].set_xlabel("alt tasiyici"); ax[0].set_ylabel("ortalama genlik"); ax[0].legend(); ax[0].set_title("Genlik profili")
for c in range(7): ax[1].plot(X[y == c].std(1).mean(0), lw=1, label=SINIFLAR[c])
ax[1].set_xlabel("kanal (0-29 anten1, 30-59 anten2, 60-89 anten3)"); ax[1].set_ylabel("zamansal std")
ax[1].legend(fontsize=8, ncol=2); ax[1].set_title("Harekete duyarlilik"); plt.tight_layout(); plt.show()

## 6. Kanallar arasi korelasyon
Komsu alt tasiyicilar birbirine cok benzer (yuksek korelasyon) -> 90 kanalin bir kismi **tekrar eden bilgi** tasir.
Bu, oznitelik ozetlemenin / PCA'nin ise yarayacagini gosterir.

In [ ]:
kor = np.corrcoef(X[:500].reshape(-1, 90).T)
plt.figure(figsize=(6, 5)); plt.imshow(kor, cmap="coolwarm", vmin=-1, vmax=1); plt.colorbar()
plt.title("90 kanal korelasyonu"); plt.xlabel("kanal"); plt.ylabel("kanal"); plt.show()
ust = kor[np.triu_indices(90, 1)]
print(f"Komsu kanal ort. korelasyon: {np.mean([kor[i, i + 1] for i in range(89)]):.3f} | tum ciftler: {ust.mean():.3f}")

## 7. Oznitelik cikarimi (klasik ML icin)
Klasik algoritmalar 22.500 ham sayiyla iyi calismaz; her ornegi **910 oznitelige** ozetliyoruz (`src/ozellik.py`):

| Grup | Adet | Ne olcer | Neye yarar |
|---|---|---|---|
| `ort`, `medyan` | 90+90 | kanalin ortalama seviyesi | kisinin konumu/durusu (yatma vs ayakta) |
| `std`, `aralik`, `min`, `maks` | 4x90 | ne kadar dalgalandi | hareket var mi, ne kadar buyuk |
| `enerji` | 90 | genligin karesinin ortalamasi | toplam sinyal gucu |
| `ort_mutlak_fark` | 90 | ardisik paketler arasi degisim | hareketin **hizi** (kosma > yurume > oturma) |
| `carpiklik` | 90 | dagilimin bir yone yigilmasi | tek yonlu degisim (oturma, kalkma) |
| `basiklik` | 90 | ani tek seferlik sicramalar | **dusme** gibi kisa ve sert olaylar |
| `hareket_dilim1-5` | 5 | 5 zaman diliminde hareket yogunlugu | hareket **ne zaman** oldu: surekli mi (yurume) patlama mi (dusme) |
| `fft_bant1-5` | 5 | frekans bantlarindaki guc (dusuk->yuksek) | yavas mi hizli/tekrarli mi (adim ritmi) |

In [ ]:
# Oznitelikleri Drive'daki onbellekten oku; yoksa hesapla ve kaydet (~1 dk)
ONBELLEK = f"{ortak.PROJE_YOLU}/data/ozellik_uthar.npz"
if os.path.exists(ONBELLEK):
    d = np.load(ONBELLEK, allow_pickle=True)
    F = {k: d[k] for k in ["F_train", "F_val", "F_test", "y_train", "y_val", "y_test"]}
    isimler = list(d["isimler"])
    print("Onbellekten okundu:", ONBELLEK)
else:
    F = {}
    for b in ["train", "val", "test"]:
        F[f"F_{b}"], isimler = ozellik.ozellik_cikar(v[f"X_{b}"])
        F[f"y_{b}"] = v[f"y_{b}"]
    np.savez_compressed(ONBELLEK, isimler=np.array(isimler), **F)
    print("Hesaplandi ve kaydedildi:", ONBELLEK)
print({k: x.shape for k, x in F.items()})

## 8. Hangi oznitelikler ayirt edici?
**ANOVA F-skoru:** bir oznitelikte siniflarin ortalamalari birbirinden ne kadar farkli (yuksek = ayirt edici).
**Random Forest onemi:** agaclarin karar verirken o oznitelige ne kadar dayandigi. Ayrica gruplara (istatistik turu, anten) topluyoruz.

In [ ]:
from sklearn.feature_selection import f_classif
from sklearn.ensemble import RandomForestClassifier
Ftr, ytr = F["F_train"], F["y_train"]
fs, _ = f_classif(Ftr, ytr); fs = np.nan_to_num(fs)
sira = np.argsort(fs)[::-1][:20]
display(pd.DataFrame({"oznitelik": [isimler[i] for i in sira], "F_skoru": fs[sira].round(1)}))

rf = RandomForestClassifier(300, class_weight="balanced", n_jobs=-1, random_state=ortak.SEED).fit(Ftr, ytr)
tur, anten = ozellik.grup_ozet(isimler, rf.feature_importances_)
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
s = pd.Series(tur).sort_values(); s.plot.barh(ax=ax[0], title="RF onemi - oznitelik turu")
pd.Series(anten).plot.bar(ax=ax[1], rot=0, title="RF onemi - anten"); plt.tight_layout(); plt.show()

In [ ]:
# Ozet oznitelikler sinif bazinda (tum kanallarin ortalamasi)
def grup_ort(ad):
    idx = [i for i, n in enumerate(isimler) if n.endswith("__" + ad)]
    return Ftr[:, idx].mean(1)
ozet = {"std (hareket buyuklugu)": grup_ort("std"), "ort_mutlak_fark (hiz)": grup_ort("ort_mutlak_fark"),
        "basiklik (ani sicrama)": grup_ort("basiklik"), "fft_bant5 (yuksek frekans)": Ftr[:, isimler.index("fft_bant5")]}
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
for a_, (ad, deger) in zip(ax, ozet.items()):
    a_.boxplot([deger[ytr == c] for c in range(7)], showfliers=False)
    a_.set_xticks(range(1, 8), SINIFLAR, rotation=45); a_.set_title(ad)
plt.tight_layout(); plt.show()

## 9. PCA ile 2 boyutta gorunum
910 oznitelik standartlastirilip 2 boyuta indirgenir. Siniflar ayri bulutlar olusturuyorsa klasik ML iyi calisir;
ust uste biniyorsa (ornegin oturma-kalkma) derin modellere ihtiyac oldugunu gosterir.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
olc = StandardScaler().fit(Ftr)
pca = PCA(50, random_state=ortak.SEED).fit(olc.transform(Ftr))
Z = pca.transform(olc.transform(Ftr))
print(f"Ilk 2 bilesen varyans: %{100 * pca.explained_variance_ratio_[:2].sum():.1f} | ilk 50: %{100 * pca.explained_variance_ratio_.sum():.1f}")
plt.figure(figsize=(7, 5.5))
for c in range(7): plt.scatter(Z[ytr == c, 0], Z[ytr == c, 1], s=6, alpha=.6, label=SINIFLAR[c])
plt.legend(markerscale=3, fontsize=8); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("PCA (egitim)"); plt.show()

## 10. Rapora yazilacak notlar
Bu bolumu grafiklere bakarak doldur:
- Veri: 4973 ornek (3977/496/500), her biri 250x90, eksik deger yok.
- En ayirt edici oznitelik turleri: ...
- En onemli anten: ...
- Dusme hangi oznitelikte belirgin: ...
- PCA'da karisan siniflar: ...

Sonraki adim: herkes `H2_ML_sablon.ipynb`'yi kendi algoritmasiyla calistirir.